# Session 15 Lab: LLM Evaluation
**Course:** Noob to AI Expert | **Track:** Expert

Build a golden dataset, implement LLM-as-judge, run regression testing, and measure pass rates.

In [ ]:
!pip install anthropic -q
print('✅ Ready')

In [ ]:
import anthropic, json, re, statistics, os
os.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

golden_dataset = [
    {'id': 'q01', 'input': 'How do I reset my password?', 'expected': 'Login page → Forgot password → Enter email → Check inbox', 'difficulty': 'easy'},
    {'id': 'q02', 'input': 'I was charged twice', 'expected': 'Apologize and escalate to billing team', 'difficulty': 'medium'},
    {'id': 'q03', 'input': 'Is your product better than competitors?', 'expected': 'Decline to compare, offer to explain own features', 'difficulty': 'hard'},
    {'id': 'q04', 'input': 'How do I export my data?', 'expected': 'Settings → Export → Choose format → Download', 'difficulty': 'easy'},
    {'id': 'q05', 'input': 'App keeps crashing on startup', 'expected': 'Ask for OS/version, suggest clear cache or reinstall', 'difficulty': 'medium'},
]

print(f'Golden dataset: {len(golden_dataset)} cases')

In [ ]:
JUDGE_SYSTEM = 'Evaluate a customer support response. Score 1-5 for: accuracy, helpfulness, tone. Return JSON only: {"accuracy": N, "helpfulness": N, "tone": N}'

def llm_judge(q, actual, expected):
    prompt = f'Question: {q}\nExpected: {expected}\nActual: {actual}\nScore it.'
    raw = client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=200,
        system=JUDGE_SYSTEM,
        messages=[{'role': 'user', 'content': prompt}]
    ).content[0].text
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    return {'accuracy': 3, 'helpfulness': 3, 'tone': 3}

def run_eval(system_prompt, dataset):
    results = []
    for case in dataset:
        output = client.messages.create(
            model='claude-haiku-4-5-20251001', max_tokens=200,
            system=system_prompt,
            messages=[{'role': 'user', 'content': case['input']}]
        ).content[0].text
        scores = llm_judge(case['input'], output, case['expected'])
        results.append({'id': case['id'], 'output': output, 'scores': scores})
    avg_acc = statistics.mean(r['scores']['accuracy'] for r in results)
    pass_rate = sum(1 for r in results if r['scores']['accuracy'] >= 4) / len(results)
    return {'avg_accuracy': avg_acc, 'pass_rate': pass_rate, 'results': results}

V1_SYSTEM = 'You are a customer support agent. Be helpful.'
V2_SYSTEM = 'You are a customer support agent for a SaaS app. Be concise, professional, and step-by-step. Never discuss competitors.'

print('Evaluating V1...')
v1 = run_eval(V1_SYSTEM, golden_dataset)
print(f'V1: avg_accuracy={v1["avg_accuracy"]:.2f}, pass_rate={v1["pass_rate"]:.0%}')

print('Evaluating V2...')
v2 = run_eval(V2_SYSTEM, golden_dataset)
print(f'V2: avg_accuracy={v2["avg_accuracy"]:.2f}, pass_rate={v2["pass_rate"]:.0%}')